Sanskrit → English — Fine-tuning IndicTrans2 (LoRA)
Fine-tunes the pretrained IndicTrans2 model on the provided 10k parallel corpus using LoRA (parameter-efficient — trainable on a single GPU), then evaluates BLEU / BERTScore / efficiency and writes submission.csv. Fine-tuning on in-domain data is the main lever to lift BLEU on this corpus (scriptural + programming-tutorial sentences).

Pre-trained model disclosure (required). Base model: ai4bharat/indictrans2-indic-en-1B (Gala et al., 2023), fine-tuned with LoRA on the provided training data only. roberta-large is used only inside the mandated bert-score metric. No external data, no external APIs.

Verified vs. run-on-your-GPU. The toolkit API, version pins, LoRA config, and data flow were verified; the training loop runs on your GPU (the 1B model can't load in this sandbox). If a tokenization detail differs in a newer toolkit release, cross-check the official example: https://github.com/AI4Bharat/IndicTrans2/tree/main/huggingface_interface.

Version note. Requires transformers==4.44.2 (5.x breaks IndicTransToolkit). Efficiency trade-off: the 1B base scores near-zero on the params/time terms; set the distilled 200M model below for a far better efficiency/quality balance.

In [1]:
!pip -q install torch "transformers==4.44.2"  "tokenizers==0.19.1" \
    "peft==0.13.2"  "accelerate==0.34.2"\
    IndicTransToolkit bert-score sacrebleu nltk pandas datasets

!pip -q uninstall -y torchao
import nltk; nltk.download('punkt', quiet=True)
print("Deps installed. Use Runtime > Run all (top to bottom). If you hit a stale-version error, Restart session once and Run all again.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.3/546.3 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

1· HuggingFace authentication (gated repo)
Accept access once at https://huggingface.co/ai4bharat/indictrans2-indic-en-1B, create a Read token, and store it as a Colab secret HF_TOKEN . Never hardcode it — this notebook is public.

In [12]:
from huggingface_hub import login
import os
token = "hf_xxxxxxxxxxxxxxxxx"
try:
    from google.colab import userdata; token = userdata.get("HF_TOKEN")
except Exception: pass
token = token or os.environ.get("HF_TOKEN")
if not token:
    import getpass; token = getpass.getpass("Enter HF token (hidden): ")
login(token=token); print("HF login OK")

HF login OK


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!cp -r /content/drive/MyDrive/Data_to_Students /content/

In [9]:
import os

print(os.getcwd())
print(os.listdir())
print(os.listdir("Data_to_Students"))

/content
['.config', 'Data_to_Students', 'drive', 'sample_data']
['train_en_10000.csv', 'test_en_1000.csv', 'train_sa_10000.csv', 'dev_sa_1000.csv', 'dev_en_1000.csv', 'test_sa_1000.csv']


In [3]:
import torch
DATA_DIR = "Data_to_Students"
MODEL    = "ai4bharat/indictrans2-indic-en-dist-200M"
SRC_LANG, TGT_LANG = "san_Deva", "eng_Latn"
MAXLEN   = 256
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.05
EPOCHS, LR = 20, 2e-4
TRAIN_BS, EVAL_BS, GRAD_ACCUM = 32, 64, 1
BEAMS = 8
OUT_DIR = "indictrans2-sa-en-lora"
print("device:", DEVICE, "| base model:", MODEL)

device: cuda | base model: ai4bharat/indictrans2-indic-en-dist-200M


In [10]:
import os, glob, pandas as pd
def _read(p): return pd.read_csv(p, encoding="utf-8-sig")
def load_split(split):
    n = "10000" if split == "train" else "1000"
    sa=_read(os.path.join(DATA_DIR,f"{split}_sa_{n}.csv")); en=_read(os.path.join(DATA_DIR,f"{split}_en_{n}.csv"))
    sa.columns=[c.strip() for c in sa.columns]; en.columns=[c.strip() for c in en.columns]
    df=sa.merge(en,on="Source_id",how="inner")
    df["Sentence_sa"]=df["Sentence_sa"].fillna("").astype(str).str.strip()
    df["Sentence_en"]=df["Sentence_en"].fillna("").astype(str).str.strip()
    return df[(df.Sentence_sa!="")&(df.Sentence_en!="")].reset_index(drop=True)
def load_test(filename=None):
    if filename is None:
        path=os.path.join(DATA_DIR,"test_sa_1000.csv")
        if not os.path.exists(path):
            h=sorted(glob.glob(os.path.join(DATA_DIR,"*test*sa*.csv"))); path=h[0] if h else path
    else: path=filename if os.path.exists(filename) else os.path.join(DATA_DIR,filename)
    sa=_read(path); sa.columns=[c.strip() for c in sa.columns]
    sa["Sentence_sa"]=sa["Sentence_sa"].fillna("").astype(str).str.strip(); return sa
train_df, dev_df = load_split("train"), load_split("dev")
print("train", len(train_df), "| dev", len(dev_df))

train 10000 | dev 1000


In [13]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor, IndicDataCollator
from peft import LoraConfig, get_peft_model

DTYPE = torch.bfloat16 if DEVICE=="cuda" else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=DTYPE)
ip = IndicProcessor(inference=False)

lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                  bias="none", task_type="SEQ_2_SEQ_LM", target_modules="all-linear")
model = get_peft_model(model, lora)
model.enable_input_require_grads()
model.print_trainable_parameters()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

trainable params: 25,952,256 || all params: 237,732,864 || trainable%: 10.9166


In [14]:
from datasets import Dataset
def to_features(df):
    src = ip.preprocess_batch(df["Sentence_sa"].tolist(), src_lang=SRC_LANG, tgt_lang=TGT_LANG)
    enc = tokenizer(src, text_target=df["Sentence_en"].tolist(),
                    truncation=True, max_length=MAXLEN)
    return Dataset.from_dict({"input_ids": enc["input_ids"],
                              "attention_mask": enc["attention_mask"],
                              "labels": enc["labels"]})
train_ds = to_features(train_df)
eval_ds  = to_features(dev_df)

import IndicTransToolkit.collator as _col
from transformers.data.data_collator import pad_without_fast_tokenizer_warning as _pad
_col.pad_without_fast_tokenizer_warning = _pad

collator = IndicDataCollator(tokenizer=tokenizer, model=model,
                             padding="longest", label_pad_token_id=-100)
print("tokenized:", len(train_ds), "train /", len(eval_ds), "dev")

tokenized: 10000 train / 1000 dev


Inference + metric helpers
Defined before training so BLEU and BERTScore can be measured after every epoch. roberta-large is loaded once via a persistent BERTScorer.

In [15]:
import time
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from bert_score import BERTScorer

LEN_PENALTY = 1.0
ip_inf = IndicProcessor(inference=True)
_bert_scorer = BERTScorer(lang="en", rescale_with_baseline=True)

@torch.no_grad()
def translate(sentences, bs=EVAL_BS, beams=BEAMS, len_pen=None):
    model.eval()
    lp = LEN_PENALTY if len_pen is None else len_pen
    out = []
    for i in range(0, len(sentences), bs):
        chunk = ip_inf.preprocess_batch(sentences[i:i+bs], src_lang=SRC_LANG, tgt_lang=TGT_LANG)
        enc = tokenizer(chunk, truncation=True, padding="longest",
                        return_tensors="pt", max_length=MAXLEN).to(DEVICE)
        gen = model.generate(**enc, num_beams=beams, max_length=MAXLEN, num_return_sequences=1,
                             length_penalty=lp, no_repeat_ngram_size=3, early_stopping=True)
        dec = tokenizer.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        out += ip_inf.postprocess_batch(dec, lang=TGT_LANG)
    return out

def bleu(refs, hyps):
    return corpus_bleu([[r.lower().split()] for r in refs], [h.lower().split() for h in hyps],
                       smoothing_function=SmoothingFunction().method1)
def bertf1(refs, hyps):
    P, R, F = _bert_scorer.score(hyps, refs); return float(F.mean())

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Train (LoRA)

In [16]:
import IndicTransToolkit.collator as _col
from transformers.data.data_collator import pad_without_fast_tokenizer_warning as _pad
_col.pad_without_fast_tokenizer_warning = _pad
_base_collator = IndicDataCollator(tokenizer=tokenizer, model=model, padding="longest", label_pad_token_id=-100)

def _shift_right(labels, pad_id, start_id):
    dec = labels.new_zeros(labels.shape); dec[:, 1:] = labels[:, :-1].clone(); dec[:, 0] = start_id
    dec.masked_fill_(dec == -100, pad_id); return dec

class _DecCollator:
    def __init__(self, base, model):
        self.base = base; cfg = model.config
        self.pad_id = cfg.pad_token_id if cfg.pad_token_id is not None else tokenizer.pad_token_id
        self.start_id = cfg.decoder_start_token_id if cfg.decoder_start_token_id is not None else (getattr(cfg, "eos_token_id", None) or self.pad_id)
    def __call__(self, features, return_tensors=None):
        b = self.base(features)
        if "labels" in b and "decoder_input_ids" not in b:
            labels = b["labels"]
            if not torch.is_tensor(labels): labels = torch.as_tensor(labels); b["labels"] = labels
            b["decoder_input_ids"] = _shift_right(labels, self.pad_id, self.start_id)
        return b

collator = _DecCollator(_base_collator, model)
_chk = collator([train_ds[0], train_ds[1]])
assert "decoder_input_ids" in _chk, "collator did not add decoder_input_ids"
print("collator OK | decoder_input_ids", tuple(_chk["decoder_input_ids"].shape), "| decoder_start_id", collator.start_id)

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback
from peft import get_peft_model_state_dict, set_peft_model_state_dict
import copy, pandas as pd

class EpochScorer(TrainerCallback):
    """After each epoch: compute the assignment metrics (BLEU + BERTScore) on dev,
       keep the best-by-BLEU adapter in memory, and early-stop on a BLEU plateau."""
    def __init__(self, patience=4):
        self.best=-1.0; self.wait=0; self.patience=patience; self.history=[]; self.best_sd=None
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        ep = int(round(state.epoch))
        refs = dev_df["Sentence_en"].tolist()
        hyp  = translate(dev_df["Sentence_sa"].tolist())
        b = bleu(refs, hyp)
        try: bs = bertf1(refs, hyp)
        except Exception as e: bs = float("nan")
        self.history.append({"epoch": ep, "BLEU": round(b,4), "BERTScore": round(bs,4)})
        star = ""
        if b > self.best:
            self.best = b; self.wait = 0
            self.best_sd = copy.deepcopy(get_peft_model_state_dict(model)); star = "  <-- best"
        else:
            self.wait += 1
        print(f"\n>>> Epoch {ep}: dev BLEU={b:.4f} | dev BERTScore F1={bs:.4f}{star}\n")
        if self.wait >= self.patience:
            print(f"    BLEU has not improved for {self.patience} epochs -> stopping early.")
            control.should_training_stop = True
        model.train()
        return control

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR, num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.05, weight_decay=0.01,
    lr_scheduler_type="cosine", label_smoothing_factor=0.1,
    bf16=(DEVICE=="cuda"), logging_steps=25,
    eval_strategy="no", save_strategy="no",
    report_to="none", label_names=["labels"], predict_with_generate=False,
    remove_unused_columns=False)

scorer_cb = EpochScorer(patience=4)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
                         data_collator=collator, tokenizer=tokenizer, callbacks=[scorer_cb])
trainer.train()

if scorer_cb.best_sd is not None:
    set_peft_model_state_dict(model, scorer_cb.best_sd)
    print(f"\nRestored best-by-BLEU adapter (dev BLEU {scorer_cb.best:.4f}).")
model.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
print("\nPer-epoch scores:"); print(pd.DataFrame(scorer_cb.history).to_string(index=False))

collator OK | decoder_input_ids (2, 10) | decoder_start_id 2


Step,Training Loss
25,4.455400
50,4.065900
75,3.581300
100,3.381800
125,3.230300
150,3.152500
175,3.038600
200,3.161600
225,3.131100
250,3.116200



>>> Epoch 1: dev BLEU=0.2149 | dev BERTScore F1=0.5554  <-- best


>>> Epoch 2: dev BLEU=0.2341 | dev BERTScore F1=0.5730  <-- best


>>> Epoch 3: dev BLEU=0.2401 | dev BERTScore F1=0.5793  <-- best


>>> Epoch 4: dev BLEU=0.2455 | dev BERTScore F1=0.5839  <-- best


>>> Epoch 5: dev BLEU=0.2492 | dev BERTScore F1=0.5887  <-- best


>>> Epoch 6: dev BLEU=0.2493 | dev BERTScore F1=0.5916  <-- best


>>> Epoch 7: dev BLEU=0.2478 | dev BERTScore F1=0.5890


>>> Epoch 8: dev BLEU=0.2538 | dev BERTScore F1=0.5918  <-- best


>>> Epoch 9: dev BLEU=0.2501 | dev BERTScore F1=0.5904


>>> Epoch 10: dev BLEU=0.2524 | dev BERTScore F1=0.5926


>>> Epoch 11: dev BLEU=0.2537 | dev BERTScore F1=0.5894


>>> Epoch 12: dev BLEU=0.2539 | dev BERTScore F1=0.5940  <-- best


>>> Epoch 13: dev BLEU=0.2531 | dev BERTScore F1=0.5912


>>> Epoch 14: dev BLEU=0.2520 | dev BERTScore F1=0.5923


>>> Epoch 15: dev BLEU=0.2528 | dev BERTScore F1=0.5915


>>> Epoch 16: dev BLEU=0.2525 | dev BERTScore F1=0.5936

 

Evaluate (BLEU, BERTScore, efficiency)

In [17]:
model = model.merge_and_unload()
model.config.use_cache = True
model.eval()
EVAL_BS = 128
print("optimized for inference")

optimized for inference


In [18]:
import time
dev_refs = dev_df["Sentence_en"].tolist()
t0 = time.time(); dev_hyp = translate(dev_df["Sentence_sa"].tolist()); dt = time.time()-t0
n_par = sum(p.numel() for p in model.parameters())
print(f"FINAL Dev BLEU         : {bleu(dev_refs, dev_hyp):.4f}")
print(f"FINAL Dev BERTScore F1 : {bertf1(dev_refs, dev_hyp):.4f}")
print(f"Dev inference time     : {dt:.1f}s ({dt/len(dev_df)*1000:.0f} ms/sent)")
print(f"Total params           : {n_par:,}")

FINAL Dev BLEU         : 0.2553
FINAL Dev BERTScore F1 : 0.5943
Dev inference time     : 154.2s (154 ms/sent)
Total params           : 211,780,608


In [19]:
test_df=load_test()
t0=time.time(); test_hyp=translate(test_df["Sentence_sa"].tolist()); tt=time.time()-t0
pd.DataFrame({"Source_id":test_df["Source_id"],"Sentence_en":test_hyp}).to_csv(
    "submission.csv",index=False,encoding="utf-8")
print(f"wrote submission.csv | {len(test_df)} rows | {tt:.1f}s ({tt/len(test_df)*1000:.0f} ms/sent)")
gp=os.path.join(DATA_DIR,"test_en_1000.csv")
if os.path.exists(gp):
    g=_read(gp); g.columns=[c.strip() for c in g.columns]
    m=test_df.merge(g,on="Source_id"); refs=m["Sentence_en"].fillna("").astype(str).tolist()
    print(f"Public-test BLEU     : {bleu(refs,test_hyp):.4f}")
    try: print(f"Public-test BERTScore: {bertf1(refs,test_hyp):.4f}")
    except Exception as e: print("BERTScore:", e)

wrote submission.csv | 1000 rows | 154.6s (155 ms/sent)
Public-test BLEU     : 0.2455
Public-test BERTScore: 0.5934


In [20]:
for i in range(min(8,len(dev_df))):
    print("SA :",dev_df["Sentence_sa"].iloc[i]); print("REF:",dev_refs[i]); print("HYP:",dev_hyp[i]); print()

SA : ते वीराः ।
REF: Those are brave men.
HYP: Those are heroes.

SA : 'इन्फ़ैनेट् लूप्' इतीदं व्यवस्थां निरुत्तरां कारयति ।
REF: Infinite loop  can cause the system to become unresponsive.
HYP: An infinite loop brings the system to an idle state.

SA : ततस्तस्य गात्रे निष्ठीवं दत्वा तेन वेत्रेण शिर आजघ्नुः।
REF: "And they spit upon him, and took the reed, and smote him on the head."
HYP: "And they laid a snare on him and he beheaded him with a reed.

SA : एते तिथी ।
REF: These two are dates.
HYP: These are dates.

SA : "बहुविचारेषु जातषु पितर उत्थाय कथितवान्, हे भ्रातरो यथा भिन्नदेशीयलोका मम मुखात् सुसंवादं श्रुत्वा विश्वसन्ति तदर्थं बहुदिनात् पूर्व्वम् ईश्वरोस्माकं मध्ये मां वृत्वा नियुक्तवान्।"
REF: "And when there had been much disputing, Peter rose up, and said unto them, Men and brethren, ye know how that a good while ago God made choice among us, that the Gentiles by my mouth should hear the word of the gospel, and believe."
HYP: "And after much deliberation Peter arose and said

In [21]:
!zip -r /content/indictrans2-sa-en-lora.zip /content/indictrans2-sa-en-lora

  adding: content/indictrans2-sa-en-lora/ (stored 0%)
  adding: content/indictrans2-sa-en-lora/dict.SRC.json (deflated 80%)
  adding: content/indictrans2-sa-en-lora/model.SRC (deflated 61%)
  adding: content/indictrans2-sa-en-lora/dict.TGT.json (deflated 70%)
  adding: content/indictrans2-sa-en-lora/special_tokens_map.json (deflated 79%)
  adding: content/indictrans2-sa-en-lora/adapter_config.json (deflated 53%)
  adding: content/indictrans2-sa-en-lora/adapter_model.safetensors (deflated 7%)
  adding: content/indictrans2-sa-en-lora/model.TGT (deflated 53%)
  adding: content/indictrans2-sa-en-lora/tokenizer_config.json (deflated 71%)
  adding: content/indictrans2-sa-en-lora/README.md (deflated 66%)


In [23]:
import os

print(os.path.exists("/content/indictrans2-sa-en-lora"))
print(os.listdir("/content/indictrans2-sa-en-lora"))

True
['dict.SRC.json', 'model.SRC', 'dict.TGT.json', 'special_tokens_map.json', 'adapter_config.json', 'adapter_model.safetensors', 'model.TGT', 'tokenizer_config.json', 'README.md']


In [24]:
from google.colab import files
files.download("/content/indictrans2-sa-en-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
from google.colab import files

files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Private test set (evaluation, 5 July)

In [22]:
PRIVATE_TEST_SA="test_sa_1000.csv"
ev=load_test(PRIVATE_TEST_SA)
hyp=translate(ev["Sentence_sa"].tolist())
pd.DataFrame({"Source_id":ev["Source_id"],"Sentence_en":hyp}).to_csv("submission.csv",index=False,encoding="utf-8")
print("wrote submission.csv |",len(ev),"rows")

wrote submission.csv | 1000 rows


Reload the fine-tuned adapter later (no retraining)

In [27]:
from peft import PeftModel
base = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, OUT_DIR).to(DEVICE).eval()